# Phân Tích Dữ Liệu Khảo Sát Tính Khả Thi - Dự Đoán Khả Năng Bán Chạy Trên Sàn TMĐT

## 1. Phát Biểu Bài Toán

### Đề Tài
Phân tích dữ liệu khảo sát tính khả thi cho việc xây dựng mô hình dự đoán khả năng bán chạy của sản phẩm trên sàn thương mại điện tử (Tiki.vn).

### Mục Tiêu
Khảo sát mối quan hệ giữa các đặc trưng của sản phẩm và lượng hàng bán ra để xác định xem dữ liệu hiện có đủ cơ sở để xây dựng mô hình dự báo hay không.

### Xác Định Biến Mục Tiêu ($Y$)
Biến mục tiêu là **sold** (số lượng đã bán).

#### Trường Hợp 1: Hồi Quy (Regression)
Nếu $Y$ giữ nguyên là số thực/số nguyên (số lượng bán cụ thể):
$$\text{sold} \in \mathbb{Z}^+ \text{ (số lượng bán tuyệt đối)}$$

#### Trường Hợp 2: Phân Lớp (Classification) - *Hướng Tối Ưu*
Chia $Y$ thành các nhóm dựa trên ngưỡng cụ thể:
$$Y = \begin{cases}
1 & \text{nếu sold} > \text{threshold (Bán chạy)} \\
0 & \text{nếu sold} \leq \text{threshold (Bán chậm)}
\end{cases}$$

### Các Biến Độc Lập ($X_i$)
Tổng cộng 7 biến độc lập được thu thập từ quá trình crawl dữ liệu Tiki.vn:

| Biến | Kiểu Dữ Liệu | Mô Tả |
|------|-------------|-------|
| `product_name` | Danh mục (Categorical) | Tên sản phẩm (có thể trích xuất các đặc điểm ngôn ngữ) |
| `price` | Định lượng (Numerical) | Giá bán của sản phẩm (VNĐ) |
| `discount_percent` | Định lượng (Numerical) | Phần trăm giảm giá (%) |
| `product_rating` | Định lượng (Numerical) | Đánh giá sao từ khách hàng (0-5) |
| `number_of_reviews` | Định lượng (Numerical) | Số lượng bình luận/đánh giá |
| `freeship` | Nhị phân (Binary) | Có hỗ trợ miễn phí vận chuyển (0/1) |
| `category` | Danh mục (Categorical) | Danh mục sản phẩm (điện thoại, laptop, ...) |

---

## 2. Xác Định Loại Hình Bài Toán Mô Hình Hóa

### Phân Loại Bài Toán

Dựa trên biến mục tiêu **sold**, bài toán này được xác định như sau:

#### **Lựa Chọn Chính: Bài Toán Phân Lớp (Classification)**

Chúng tôi sử dụng phân lớp nhị phân vì:
- Dễ diễn giải kết quả cho các bên liên quan
- Tương ứng với hành động kinh doanh cụ thể (tập trung vào sản phẩm bán chạy)
- Dữ liệu thường không cân bằng tự nhiên, có thể xử lý được

**Định nghĩa lớp:**
- **Nhãn 1: Bán Chạy** - Số lượng bán vượt quá ngưỡng kỳ vọng
- **Nhãn 0: Bán Chậm / Bán Tầm Thường** - Số lượng bán ở mức bình thường hoặc dưới kỳ vọng

$$Y_{\text{binary}} = \begin{cases}
1 & \text{(Bán chạy)} \quad \text{nếu} \quad \text{sold} > Q_3 \text{ hoặc sold} > \mu + \sigma \\
0 & \text{(Bán chậm)} \quad \text{nếu} \quad \text{sold} \leq Q_3 \text{ hoặc sold} \leq \mu + \sigma
\end{cases}$$

Trong đó:
- $Q_3$ là tứ phân vị thứ 3 (75th percentile)
- $\mu$ là giá trị trung bình của `sold`
- $\sigma$ là độ lệch chuẩn

#### **Lựa Chọn Phụ: Bài Toán Hồi Quy (Regression)**

Nếu cần dự báo số lượng bán chính xác:
$$\hat{\text{sold}} = f(X_1, X_2, ..., X_7) \in \mathbb{Z}^+$$

---

## 3. Tính Khả Thi của Bài Toán (Về Mặt Dữ Liệu)

Bài toán này **hoàn toàn khả thi** để thực hiện mô hình hóa vì những lý do sau:

### 3.1 Tính Tương Quan Logic
Các đặc trưng có mối liên hệ nhân quả rõ rệt với biến mục tiêu:

| Đặc Trưng | Mối Quan Hệ Với Lượng Bán | Giải Thích |
|-----------|------------------------|-----------|
| `price` | Tương quan âm | Giá cao hạn chế lượng khách mua, nhưng có thể tăng khoảng lợi nhuận |
| `discount_percent` | Tương quan dương | Giảm giá kích thích nhu cầu mua hàng |
| `product_rating` | Tương quan dương | Đánh giá cao tăng độ tin tưởng của khách |
| `number_of_reviews` | Tương quan dương rất mạnh | Sản phẩm bán chạy sẽ có nhiều bình luận |
| `freeship` | Tương quan dương | Miễn phí vận chuyển là yếu tố "kích cầu" quan trọng tại VN |
| `category` | Tương quan có điều kiện | Các ngành hàng khác nhau có tốc độ bán khác nhau |

### 3.2 Sự Đa Dạng Của Đặc Trưng
- **Biến định lượng**: `price`, `discount_percent`, `product_rating`, `number_of_reviews`
- **Biến định danh**: `product_name`, `category`
- **Biến nhị phân**: `freeship`

Tập hợp này cho phép áp dụng:
- Kỹ thuật tiền xử lý đa dạng (scaling, encoding, feature engineering)
- Các thuật toán máy học mạnh mẽ (Logistic Regression, Random Forest, Gradient Boosting, Neural Networks)

### 3.3 Khả Năng Thu Thập Dữ Liệu
- Dữ liệu được thu thập thông qua Selenium webdriver từ Tiki.vn
- Có khả năng để lấy được mẫu dữ liệu rất lớn (hàng nghìn sản phẩm) từ nhiều danh mục
- Dữ liệu độc lập, không bị yếu tố thời gian (temporal dependency) cưỡng ép quá nhiều

### 3.4 Đủ Điều Kiện Thống Kê
- Số mẫu dự kiến: > 1.500 sản phẩm × 31 danh mục = **~1.500+ mẫu**
- Tỷ lệ mẫu/biến: $\frac{1500}{7} \approx 214$ (yêu cầu tối thiểu 10, đây là rất tốt)
- Động lực học: Các biến độc lập không hoàn toàn tương quan với nhau, có thể học được

---

## 4. Tập Các Đặc Trưng Hữu Ích ($X_i$) Để Xây Dựng Mô Hình

### Bảng Tóm Tắt Tầm Quan Trọng Của Từng Đặc Trưng

| Thứ Tự | Đặc Trưng | Vai Trò Trong Mô Hình | Mức Độ Quan Trọng | Ghi Chú |
|--------|-----------|-------------------|-------------------|---------|
| 1 | `number_of_reviews` | Tương quan rất mạnh với lượng bán (sản phẩm bán chạy → nhiều review) | ⭐⭐⭐⭐⭐ | Có thể là biến dự báo tốt nhất |
| 2 | `freeship` | Yếu tố "kích cầu" vô cùng quan trọng tại thị trường Việt Nam | ⭐⭐⭐⭐⭐ | Mạnh bằng rating nếu không mạnh hơn |
| 3 | `product_rating` | Quyết định tỷ lệ chuyển đổi khách xem → khách mua | ⭐⭐⭐⭐ | Rất quan trọng cho mô hình phân lớp |
| 4 | `price` | Phản ánh tính cạnh tranh về giá của sản phẩm | ⭐⭐⭐⭐ | Tương quan âm nhưng có giá trị thông tin |
| 5 | `discount_percent` | Kích thích tâm lý tiêu dùng, tăng khả năng mua | ⭐⭐⭐⭐ | Thường có lợi tích hợp với giá |
| 6 | `category` | Giúp mô hình hiểu đặc thù từng ngành hàng | ⭐⭐⭐ | Cần encoding (one-hot hoặc label) |
| 7 | `product_name` | Có thể trích xuất tính năng từ text (nhưng phức tạp) | ⭐⭐ | Tùy chọn, có thể bỏ qua ở bước đầu |

### Các Bước Tiền Xử Lý Dữ Liệu Đề Xuất

#### **Xử Lý Biến Định Lượng**
```
• price: Scale với StandardScaler hoặc MinMaxScaler
• discount_percent: Kiểm tra phạm vi [0%, 100%], xử lý ngoại lệ
• product_rating: Thường 0-5 sao, kiểm tra dữ liệu thiếu
• number_of_reviews: Xử lý giá trị 0, có thể log-transform nếu cần
```

#### **Xử Lý Biến Danh Mục**
```
• category: One-hot encoding (31 danh mục) hoặc Label encoding
• freeship: Đã ở dạng nhị phân (0/1), không cần xử lý
• product_name: Nên loại bỏ hoặc trích xuất text features (TF-IDF, Word2Vec)
```

#### **Xử Lý Giá Trị Thiếu**
```
• Số liệu từ scraping có thể có missing values
• Sử dụng: mean/median imputation hoặc xóa dòng nếu thiếu quá nhiều
```

### Công Thức Mô Hình Đề Xuất

#### **Mô Hình Phân Lớp - Logistic Regression (Baseline)**
$$P(Y=1|X) = \frac{1}{1 + e^{-(\beta_0 + \sum_{i=1}^{7} \beta_i X_i)}}$$

#### **Mô Hình Phân Lớp - Random Forest (Recommended)**
$$Y_{\text{pred}} = \arg\max_c \sum_{t=1}^{T} \mathbb{1}[\text{tree}_t(X) = c]$$

Trong đó:
- $T$ = số lượng cây trong rừng
- $\mathbb{1}[\cdot]$ = hàm chỉ thị
- $c$ = lớp (0 hoặc 1)

---

## 5. Kết Luận & Hướng Đi Tiếp Theo

✅ **Bài toán khả thi** - Đủ dữ liệu, đặc trưng, và lý do khoa học để xây dựng mô hình  
✅ **Phương pháp: Phân Lớp Nhị Phân** - Dễ diễn giải, đơn giản, hiệu quả  
✅ **Mục tiêu hiệu suất**: Accuracy ≥ 75%, Precision ≥ 70%, Recall ≥ 70%  
✅ **Thời gian phát triển**: 2-3 tuần từ EDA đến deployment  

---

---



In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import random
import re

# Cấu hình Chrome
chrome_options = Options()
chrome_options.add_argument("--start-maximized")

CATEGORIES = [
    "điện thoại", "laptop", "tai nghe", "bàn phím", "chuột máy tính", "loa bluetooth", "đồng hồ thông minh",
    "camera an ninh", "máy hút bụi", "nồi chiên không dầu", "mỹ phẩm", "dưỡng da", "giày thể thao", "áo thun",
    "balo", "bình nước", "đồ chơi trẻ em", "sách kỹ năng", "đồ gia dụng", "phụ kiện xe máy",
    "túi xách", "máy lọc không khí", "máy xay sinh tố", "bếp điện", "máy sấy tóc", "bàn học", "ghế gaming",
    "máy in", "máy ảnh", "máy tính bảng"
]

def human_sleep():
    time.sleep(random.uniform(2, 4))

def convert_quantity_format(value):
    """Chuyển đổi 1k+ thành 1000+, 2k thành 2000, v.v"""
    if not value or value == "N/A":
        return value
    value = str(value).strip().lower()
    # Thay thế k+ hoặc k thành 000+ hoặc 000
    value = re.sub(r'(\d+)k\+', r'\g<1>000+', value)
    value = re.sub(r'(\d+)k(?!\+)', r'\g<1>000', value)
    return value

def convert_to_number(sold_str):
    """Chuyển đổi chuỗi số lượng bán về số để so sánh"""
    if not sold_str or sold_str == "N/A":
        return 0
    sold_str = str(sold_str).strip().lower()
    # Xóa dấu + nếu có
    sold_str = sold_str.replace("+", "")
    # Chuyển k thành 000
    if "k" in sold_str:
        sold_str = sold_str.replace("k", "000")
    try:
        return int(sold_str)
    except:
        return 0

def crawl_tiki_selenium(categories, products_per_cat=50, save_every=20, output_file="tiki.csv"):
    all_data = []
    driver = webdriver.Chrome(options=chrome_options)
    driver.get("https://tiki.vn/")
    time.sleep(3)

    for category in categories:
        print(f"đang lấy loại sản phẩm : {category}")
        # Tìm ô tìm kiếm và nhập từ khóa
        search_box = driver.find_element(By.CSS_SELECTOR, "input[type='text'][data-view-id='main_search_form_input']")
        search_box.clear()
        search_box.send_keys(category)
        search_box.send_keys(Keys.RETURN)
        time.sleep(3)
        
        # Chọn filter "bán chạy" (Best-selling)
        try:
            sort_button = driver.find_element(By.CSS_SELECTOR, "button[aria-label*='Sắp xếp']")
            sort_button.click()
            time.sleep(1)
            best_selling = driver.find_element(By.XPATH, "//*[contains(text(), 'Bán chạy')]")
            best_selling.click()
            time.sleep(2)
        except:
            pass
        
        # Chộn filter rating từ 4 sao trở lên - Người dùng tự tích
        print(f"\n{'='*60}", flush=True)
        print(f"Loại sản phẩm: {category}", flush=True)
        print(f"{'='*60}", flush=True)
        print("Hãy TÍCH vào filter sau trên trang tiki.vn:", flush=True)
        print("✓ Bán chạy (Already selected if possible)", flush=True)
        print("✓ 4 sao trở lên (Hãy tích vào nút 4 sao)", flush=True)
        print("Sau khi đã tích xong, hãy nhấn Enter để tiếp tục...", flush=True)
        input()
        print("\n✓ Bắt đầu lấy sản phẩm...\n", flush=True)
        time.sleep(2)

        # Lấy sản phẩm từ mỗi trang - 20 sản phẩm đầu tiên, rồi chuyển sang trang tiếp theo
        page = 1
        products_in_category = 0

        while products_in_category < products_per_cat:
            # Lấy URL trang hiện tại
            current_url = driver.current_url
            if "page=" not in current_url:
                page_url = f"{current_url}?page={page}"
            else:
                page_url = f"{current_url.split('page=')[0]}page={page}"
            
            driver.get(page_url)
            time.sleep(2)
            print(f"Đang lấy từ trang {page}...", flush=True)
            
            # Lấy tối đa 20 sản phẩm từ trang này
            products = driver.find_elements(By.CSS_SELECTOR, "a.product-item")
            page_product_links = []
            
            for p in products:
                link = p.get_attribute("href")
                if link and link.startswith("https://tiki.vn/"):
                    page_product_links.append(link)
                if len(page_product_links) >= 20:  # Chỉ lấy 20 sản phẩm đầu tiên từ trang
                    break
            
            print(f"Trang {page}: Tìm được {len(page_product_links)} sản phẩm", flush=True)
            
            if len(page_product_links) == 0:
                print("Không còn sản phẩm nữa, dừng crawl", flush=True)
                break
            
            # Xử lý 20 sản phẩm từ trang hiện tại
            for link in page_product_links:
                if products_in_category >= products_per_cat:
                    break
                
                driver.get(link)
                time.sleep(2)
                data = {}
                
                try:
                    data["product_name"] = driver.find_element(By.CSS_SELECTOR, "h1").text.strip()
                except NoSuchElementException:
                    print("⊗ Skip: không có product_name", flush=True)
                    continue
                    
                try:
                    data["price"] = driver.find_element(By.CSS_SELECTOR, ".product-price__current-price").text
                except NoSuchElementException:
                    print("⊗ Skip: không có price", flush=True)
                    continue
                    
                try:
                    data["discount_percent"] = driver.find_element(By.CSS_SELECTOR, ".product-price__discount-rate").text
                except NoSuchElementException:
                    data["discount_percent"] = "N/A"
                    
                try:
                    data["product_rating"] = driver.find_element(By.CSS_SELECTOR, "div.sc-1a46a934-1.dCjKzJ").text
                except NoSuchElementException:
                    data["product_rating"] = "N/A"
                    
                try:
                    data["number_of_reviews"] = driver.find_element(By.CSS_SELECTOR, "[data-view-id='pdp_main_view_review']").text
                except NoSuchElementException:
                    data["number_of_reviews"] = "N/A"
                
                # Kiểm tra: nếu không có rating VÀ reviews cùng lúc thì skip
                if data["product_rating"] == "N/A" and data["number_of_reviews"] == "N/A":
                    print("⊗ Skip: không có rating và reviews", flush=True)
                    continue
                    
                try:
                    freeship = driver.find_element(By.XPATH, "//*[contains(text(),'Freeship')]")
                    data["freeship"] = 1
                except NoSuchElementException:
                    data["freeship"] = 0
                    
                data["category"] = category
                
                try:
                    sold_text = driver.find_element(By.CSS_SELECTOR, "[data-view-id='pdp_quantity_sold']").text
                    data["sold"] = convert_quantity_format(sold_text)
                except NoSuchElementException:
                    data["sold"] = "N/A"
                
                # Lấy tất cả sản phẩm có rating hoặc reviews (không check sold = 0 vì selector có thể không chính xác)

                all_data.append(data)
                products_in_category += 1
                print(f"đã lấy được {len(all_data)} sản phẩm", flush=True)

                if len(all_data) % save_every == 0:
                    pd.DataFrame(all_data).to_csv(output_file, index=False)
                    print(f"Lưu sản phẩm vào file {output_file}", flush=True)

                if len(all_data) >= len(categories) * products_per_cat:
                    pd.DataFrame(all_data).to_csv(output_file, index=False)
                    print(f"Đã hoàn thành crawl {len(all_data)} sản phẩm!")
                    driver.quit()
                    return

                human_sleep()
            
            page += 1  # Chuyển sang trang tiếp theo
    
    pd.DataFrame(all_data).to_csv(output_file, index=False)
    print(f"Đã lưu toàn bộ {len(all_data)} sản phẩm vào {output_file}")
    driver.quit()

# Chạy hàm crawl
crawl_tiki_selenium(CATEGORIES, products_per_cat=50, save_every=20, output_file="raw.csv")